# Préparation des données — Variable cible : education
À exécuter dans Google Colab.

Uploader **adult-all.csv** avant de lancer les cellules (icône dossier à gauche, ou `files.upload()`).

## 1. Importation des bibliothèques nécessaires

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler

## 2. Chargement du dataset

In [ ]:
cols = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
        'marital-status', 'occupation', 'relationship', 'race', 'sex',
        'capital-gain', 'capital-loss', 'hours-per-week',
        'native-country', 'income']

df = pd.read_csv("adult-all.csv", names=cols, skipinitialspace=True)

## 3. Exploration et aperçu

In [ ]:
print(df.columns)
print(df.head())
print(df.tail())
print(df.describe())
print(df.info())

## 4. Nettoyage et gestion des doublons

In [ ]:
print("Nombre de doublons:", df.duplicated().sum())
df.drop_duplicates(inplace=True)
print("Nombre de doublons:", df.duplicated().sum())
print(df.shape)

## 5. Gestion des valeurs manquantes

In [ ]:
print("Valeurs manquantes:\n", df.isnull().sum())

## 6. Suppression des variables non utilisables

In [ ]:
# regroupement des 16 niveaux de education en 7 classes ordonnées
# (education devient la cible : on la simplifie avant de continuer)
low = ['Preschool', '1st-4th', '5th-6th', '7th-8th', '9th', '10th', '11th', '12th']
education_map = {
    **{niveau: '1_Sans-bac' for niveau in low},
    'HS-grad': '2_Bac',
    'Some-college': '3_Sup-non-diplome',
    'Assoc-voc': '4_Bac+2', 'Assoc-acdm': '4_Bac+2',
    'Bachelors': '5_Licence',
    'Masters': '6_Master',
    'Prof-school': '7_Doctorat-Prof', 'Doctorate': '7_Doctorat-Prof'
}
df['education'] = df['education'].map(education_map)

# education-num = simple recodage numérique de education (la cible) -> fuite de données, on le retire
# fnlwgt = poids statistique du recensement, pas une caractéristique du profil -> on le retire
df = df.drop(columns=['education-num', 'fnlwgt'])

print(df.columns.tolist())
print(df.shape)

## 7. Visualisation

In [ ]:
print(df['education'].value_counts())

plt.figure(figsize=(10, 5))
sns.countplot(x='education', data=df, order=sorted(df['education'].unique()))
plt.xticks(rotation=45)
plt.title("Distribution de la variable cible : education")
plt.show()

## 8. Transformation et encodage

In [ ]:
label_encoders = {}
for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print("\nDataset après encodage :")
print(df.head())

# correspondance texte -> nombre pour la cible education
print("Classes de la cible (education) :", label_encoders['education'].classes_)

## 9. Normalisation

In [ ]:
df_normalized = df.copy()
scaler = StandardScaler()
df_normalized[df.select_dtypes(include=[np.number]).columns] = scaler.fit_transform(
    df[df.select_dtypes(include=[np.number]).columns]
)
print(df_normalized.head())

## 10. Matrice de corrélation

In [ ]:
plt.figure(figsize=(10, 8))
corr_matrix = df_normalized.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', cbar=True)
plt.title('Matrice de corrélation')
plt.show()

target = 'education'
print("\nCorrélations avec la cible (education) :")
print(corr_matrix[target].abs().sort_values(ascending=False))

## 11. Sélection de caractéristiques

In [ ]:
correlation_threshold = 0.05
target_corr = corr_matrix[target].abs()
selected_features = target_corr[target_corr > correlation_threshold].index.tolist()
if target in selected_features:
    selected_features.remove(target)

print(f"\nCaractéristiques sélectionnées (|corr| > {correlation_threshold}) :")
print(selected_features)

## 12. Création du DataFrame final

In [ ]:
df_final = df_normalized[selected_features + [target]]
print("\nDataFrame final :", df_final.shape)
print(df_final.head())

# Sauvegarde optionnelle du résultat
# df_final.to_csv("education_prepared.csv", index=False)